In [1]:
import torch

inputs = torch.tensor(
    [[0.72, 0.45, 0.31], # Dream  (x^1)
     [0.75, 0.20, 0.55], # big    (x^2)
     [0.30, 0.80, 0.40], # and    (x^3)
     [0.85, 0.35, 0.60], # work   (x^4)
     [0.55, 0.15, 0.75], # for    (x^5)
     [0.25, 0.20, 0.85]] # it     (x^6)
)

# correspinding words
words = ['Dream', 'big', 'and', 'work', 'for', 'it']

Generating vector corresponding to the 2nd token

In [2]:
x_2 = inputs[1]  # big (x^2)
d_in = inputs.shape[1]  # number of features (3)
d_out = 2  # number of output classes (2)

Randomly initializing Q, K, V matrices

In [4]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [6]:
print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [7]:
print(W_key)

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])


In [8]:
print(W_value)

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


In [11]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(f"Query for word '{words[1]}' {query_2}")
print(f"Key for word '{words[1]}' {key_2}")
print(f"Value for word '{words[1]}' {value_2}")

Query for word 'big' tensor([0.3131, 1.0017])
Key for word 'big' tensor([0.3126, 0.6001])
Value for word 'big' tensor([0.1852, 0.6829])


Calculating Q, K, V for X using w_q, w_k, w_v

In [13]:
querys = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("Queries shape:", querys.shape)
print("Keys shape:", keys.shape)
print("Values shape:", values.shape)

print("Queries:", querys)
print("Keys:", keys)
print("Values:", values)

Queries shape: torch.Size([6, 2])
Keys shape: torch.Size([6, 2])
Values shape: torch.Size([6, 2])
Queries: tensor([[0.3494, 0.9504],
        [0.3131, 1.0017],
        [0.3198, 1.0524],
        [0.3842, 1.2000],
        [0.2561, 1.0373],
        [0.1872, 1.0034]])
Keys: tensor([[0.2789, 0.6137],
        [0.3126, 0.6001],
        [0.3143, 0.8867],
        [0.3697, 0.7536],
        [0.3392, 0.6807],
        [0.3389, 0.7549]])
Values: tensor([[0.2336, 0.5789],
        [0.1852, 0.6829],
        [0.3232, 0.7113],
        [0.2462, 0.8042],
        [0.1780, 0.7890],
        [0.1830, 0.8328]])


Attention Score of second token

In [17]:
key_2 = keys[1]  # key for word 'big'
attn_score_22 = query_2 @ key_2  # dot product of query and key
print(attn_score_22)

tensor(0.6990)


All attention scores for 2nd token

In [16]:
attn_scores_2 = query_2 @ keys.T  # dot product of query and all keys
print(attn_scores_2)

tensor([0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624])


Attention Score

In [18]:
attn_scores = querys @ keys.T  # dot product of all queries and keys
print(attn_scores)

tensor([[0.6807, 0.6795, 0.9526, 0.8454, 0.7654, 0.8359],
        [0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624],
        [0.7350, 0.7315, 1.0337, 0.9113, 0.8248, 0.9029],
        [0.8436, 0.8402, 1.1848, 1.0464, 0.9471, 1.0361],
        [0.7080, 0.7025, 1.0003, 0.8764, 0.7929, 0.8699],
        [0.6680, 0.6606, 0.9486, 0.8254, 0.7465, 0.8210]])


Scaled by 1/sqrt(d)

In [21]:
d_k = keys.shape[-1]  # dimension of key vectors
scaled_attn_scores = attn_scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
print(scaled_attn_scores)

tensor([[0.4813, 0.4805, 0.6736, 0.5978, 0.5412, 0.5911],
        [0.4964, 0.4943, 0.6977, 0.6157, 0.5572, 0.6098],
        [0.5198, 0.5172, 0.7310, 0.6444, 0.5832, 0.6384],
        [0.5965, 0.5941, 0.8378, 0.7399, 0.6697, 0.7327],
        [0.5006, 0.4967, 0.7073, 0.6197, 0.5607, 0.6151],
        [0.4723, 0.4671, 0.6708, 0.5836, 0.5278, 0.5805]])


Attention Weights

In [23]:
attn_weights = torch.softmax(scaled_attn_scores, dim=-1)
print(attn_weights)

tensor([[0.1536, 0.1534, 0.1861, 0.1725, 0.1630, 0.1714],
        [0.1531, 0.1528, 0.1873, 0.1725, 0.1627, 0.1715],
        [0.1525, 0.1521, 0.1884, 0.1728, 0.1625, 0.1717],
        [0.1505, 0.1501, 0.1915, 0.1737, 0.1619, 0.1724],
        [0.1530, 0.1524, 0.1881, 0.1724, 0.1625, 0.1716],
        [0.1538, 0.1530, 0.1875, 0.1719, 0.1625, 0.1713]])


Context Vector of 2nd token

In [24]:
context_vec_2 = attn_weights[1] @ values  # weighted sum of values
print(context_vec_2)

tensor([0.2274, 0.7362])


Context Vector

In [25]:
context_vector = attn_weights @ values
print(context_vector)

tensor([[0.2273, 0.7361],
        [0.2274, 0.7362],
        [0.2276, 0.7363],
        [0.2280, 0.7368],
        [0.2275, 0.7362],
        [0.2275, 0.7360]])


Defining everything in a Class

In [ ]:
import torch.nn as nn

class SelfAttention_V1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__() # call the constructor of the parent class (nn.Module) 
        self.W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
        self.W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
        self.W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

    def forward(self, x):
        querys = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        d_k = keys.shape[-1]  # dimension of key vectors
        attn_scores = querys @ keys.T
        attn_weights = torch.softmax(
            attn_scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32)), dim=-1
        )

        context_vector = attn_weights @ values
        return context_vector

In [33]:
torch.manual_seed(123)
sa_v1 = SelfAttention_V1(d_in=3, d_out=2)
print(sa_v1(inputs))

tensor([[0.2273, 0.7361],
        [0.2274, 0.7362],
        [0.2276, 0.7363],
        [0.2280, 0.7368],
        [0.2275, 0.7362],
        [0.2275, 0.7360]])


Usually bias is also added

In [34]:
class SelfAttention_V2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        querys = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        d_k = keys.shape[-1]  # dimension of key vectors
        attn_scores = querys @ keys.T
        attn_weights = torch.softmax(
            attn_scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32)), dim=-1
        )

        context_vector = attn_weights @ values
        return context_vector